# Project 1: Image Compression with SVD

## Introduction

Welcome to the first project of the EECS 127 Crash Course! In this notebook, we'll explore how **Singular Value Decomposition (SVD)** can be used to compress images.

### Why SVD for Image Compression?

SVD is a powerful matrix factorization technique that decomposes any matrix into three matrices:

$$A = U \Sigma V^T$$

Where:
- $A$ is an $m \times n$ matrix (our image)
- $U$ is an $m \times m$ orthogonal matrix (left singular vectors)
- $\Sigma$ is an $m \times n$ diagonal matrix (singular values)
- $V^T$ is an $n \times n$ orthogonal matrix (right singular vectors)

### The Magic of Low-Rank Approximation

By keeping only the $k$ largest singular values, we can approximate $A$:

$$A \approx A_k = U_k \Sigma_k V_k^T$$

This reduces storage from $m \times n$ to $k(m + n + 1)$ values!

### Learning Objectives

By the end of this notebook, you will:
1. Understand what SVD is and why it works for compression
2. Implement image compression using NumPy's SVD
3. Visualize the compression-quality tradeoff
4. Understand how this relates to ML concepts like dimensionality reduction

---

## Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.image import imread
from PIL import Image
import requests
from io import BytesIO

# Set random seed for reproducibility
np.random.seed(42)

# Configure matplotlib
plt.rcParams['figure.figsize'] = (15, 5)
plt.rcParams['figure.dpi'] = 100

## Part 1: Understanding SVD with a Simple Example

Before we compress images, let's understand SVD with a small matrix.

In [ ]:
# Create a simple 4x4 matrix
A = np.array([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 16]
], dtype=float)

print("Original Matrix A:")
print(A)
print(f"\nShape: {A.shape}")
print(f"Total elements: {A.size}")

In [ ]:
# Compute SVD
U, S, Vt = np.linalg.svd(A, full_matrices=True)

print("\n=== SVD Decomposition ===")
print(f"\nU shape: {U.shape}")
print(f"S shape: {S.shape}")
print(f"V^T shape: {Vt.shape}")

print("\nSingular values (sorted, descending):")
print(S)

# Reconstruct A from SVD
Sigma = np.zeros_like(A)
np.fill_diagonal(Sigma, S)
A_reconstructed = U @ Sigma @ Vt

print("\nReconstructed matrix (should match original):")
print(A_reconstructed)
print(f"\nReconstruction error: {np.linalg.norm(A - A_reconstructed):.10f}")

### Key Insight: Singular Values Indicate Importance

Notice how the singular values decrease. The larger values capture more "information" about the matrix!

In [ ]:
# Visualize singular value decay
plt.figure(figsize=(8, 4))
plt.bar(range(1, len(S)+1), S, color='steelblue', alpha=0.7)
plt.xlabel('Singular Value Index', fontsize=12)
plt.ylabel('Magnitude', fontsize=12)
plt.title('Singular Values of Matrix A', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Part 2: Low-Rank Approximation

Let's see what happens when we keep only the top $k$ singular values.

In [ ]:
def low_rank_approximation(U, S, Vt, k):
    """
    Compute rank-k approximation of a matrix from its SVD.
    
    Args:
        U: Left singular vectors (m x m)
        S: Singular values (min(m,n),)
        Vt: Right singular vectors (n x n)
        k: Number of singular values to keep
    
    Returns:
        A_k: Rank-k approximation
    """
    # Keep only first k components
    U_k = U[:, :k]
    S_k = S[:k]
    Vt_k = Vt[:k, :]
    
    # Reconstruct
    A_k = U_k @ np.diag(S_k) @ Vt_k
    
    return A_k

# Test with different ranks
for k in [1, 2, 3, 4]:
    A_k = low_rank_approximation(U, S, Vt, k)
    error = np.linalg.norm(A - A_k, 'fro')  # Frobenius norm
    
    print(f"\nRank-{k} approximation:")
    print(A_k)
    print(f"Approximation error: {error:.4f}")

## Part 3: Image Compression

Now let's apply this to real images! We'll create a sample image to work with.

In [ ]:
# Create a sample image (you can replace this with your own image)
def create_sample_image(size=512):
    """
    Create a sample grayscale image with geometric patterns.
    """
    img = np.zeros((size, size))
    
    # Add some geometric patterns
    for i in range(size):
        for j in range(size):
            img[i, j] = (np.sin(i/20) + np.cos(j/20)) * 50 + 128
    
    # Add a circle
    center = size // 2
    radius = size // 4
    y, x = np.ogrid[:size, :size]
    mask = (x - center)**2 + (y - center)**2 <= radius**2
    img[mask] = 255
    
    return img.astype(np.uint8)

# Create and display the sample image
image = create_sample_image(512)

plt.figure(figsize=(6, 6))
plt.imshow(image, cmap='gray')
plt.title('Original Image', fontsize=14, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

print(f"Image shape: {image.shape}")
print(f"Total pixels: {image.size}")
print(f"Data type: {image.dtype}")

### Load Your Own Image (Optional)

Uncomment the code below to load your own image:

In [ ]:
# # Load your own image
# from PIL import Image
# img_path = "path/to/your/image.jpg"  # Replace with your image path
# image = np.array(Image.open(img_path).convert('L'))  # Convert to grayscale
# 
# plt.figure(figsize=(6, 6))
# plt.imshow(image, cmap='gray')
# plt.title('Your Image', fontsize=14, fontweight='bold')
# plt.axis('off')
# plt.show()

## Part 4: Compress the Image

In [ ]:
# Compute SVD of the image
U_img, S_img, Vt_img = np.linalg.svd(image, full_matrices=False)

print(f"Image SVD shapes:")
print(f"U: {U_img.shape}")
print(f"S: {S_img.shape}")
print(f"Vt: {Vt_img.shape}")

# Visualize singular value spectrum
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(S_img, linewidth=2, color='steelblue')
plt.xlabel('Index', fontsize=12)
plt.ylabel('Singular Value', fontsize=12)
plt.title('Singular Values (Linear Scale)', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)

plt.subplot(1, 2, 2)
plt.semilogy(S_img, linewidth=2, color='steelblue')
plt.xlabel('Index', fontsize=12)
plt.ylabel('Singular Value (log scale)', fontsize=12)
plt.title('Singular Values (Log Scale)', fontsize=14, fontweight='bold')
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nLargest singular value: {S_img[0]:.2f}")
print(f"Smallest singular value: {S_img[-1]:.6f}")
print(f"Ratio: {S_img[0]/S_img[-1]:.2f}")

### Compress at Different Ranks

Let's compress the image using different numbers of singular values and see the results!

In [ ]:
# Test different compression levels
ranks = [5, 10, 20, 50, 100]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

# Original image
axes[0].imshow(image, cmap='gray')
axes[0].set_title('Original Image', fontsize=12, fontweight='bold')
axes[0].axis('off')

# Compressed images
for idx, k in enumerate(ranks, 1):
    # Reconstruct with rank k
    img_compressed = low_rank_approximation(U_img, S_img, Vt_img, k)
    
    # Calculate compression ratio and error
    original_size = image.size
    compressed_size = k * (U_img.shape[0] + Vt_img.shape[1] + 1)
    compression_ratio = original_size / compressed_size
    
    error = np.linalg.norm(image - img_compressed, 'fro') / np.linalg.norm(image, 'fro')
    
    # Display
    axes[idx].imshow(img_compressed, cmap='gray')
    axes[idx].set_title(f'Rank {k}\nCompression: {compression_ratio:.1f}x\nError: {error:.3f}', 
                       fontsize=10)
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## Part 5: Compression-Quality Tradeoff Analysis

In [ ]:
# Analyze compression-quality tradeoff
max_rank = min(image.shape)
ranks_to_test = range(1, max_rank, 5)

compression_ratios = []
relative_errors = []
energy_preserved = []

for k in ranks_to_test:
    # Reconstruct
    img_k = low_rank_approximation(U_img, S_img, Vt_img, k)
    
    # Compression ratio
    original_size = image.size
    compressed_size = k * (U_img.shape[0] + Vt_img.shape[1] + 1)
    compression_ratio = original_size / compressed_size
    compression_ratios.append(compression_ratio)
    
    # Relative error
    error = np.linalg.norm(image - img_k, 'fro') / np.linalg.norm(image, 'fro')
    relative_errors.append(error)
    
    # Energy preserved (sum of squared singular values)
    energy = np.sum(S_img[:k]**2) / np.sum(S_img**2)
    energy_preserved.append(energy)

# Plot tradeoff curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Compression ratio vs rank
axes[0].plot(ranks_to_test, compression_ratios, linewidth=2, color='green')
axes[0].set_xlabel('Rank k', fontsize=12)
axes[0].set_ylabel('Compression Ratio', fontsize=12)
axes[0].set_title('Compression Ratio vs Rank', fontsize=14, fontweight='bold')
axes[0].grid(alpha=0.3)

# Relative error vs rank
axes[1].plot(ranks_to_test, relative_errors, linewidth=2, color='red')
axes[1].set_xlabel('Rank k', fontsize=12)
axes[1].set_ylabel('Relative Error', fontsize=12)
axes[1].set_title('Reconstruction Error vs Rank', fontsize=14, fontweight='bold')
axes[1].grid(alpha=0.3)

# Energy preserved vs rank
axes[2].plot(ranks_to_test, [e*100 for e in energy_preserved], linewidth=2, color='blue')
axes[2].set_xlabel('Rank k', fontsize=12)
axes[2].set_ylabel('Energy Preserved (%)', fontsize=12)
axes[2].set_title('Energy Preservation vs Rank', fontsize=14, fontweight='bold')
axes[2].grid(alpha=0.3)
axes[2].axhline(y=90, color='gray', linestyle='--', alpha=0.5, label='90% energy')
axes[2].axhline(y=95, color='gray', linestyle='--', alpha=0.5, label='95% energy')
axes[2].legend()

plt.tight_layout()
plt.show()

# Find rank for 90% energy preservation
k_90 = next(i for i, e in zip(ranks_to_test, energy_preserved) if e >= 0.90)
k_95 = next(i for i, e in zip(ranks_to_test, energy_preserved) if e >= 0.95)

print(f"\nTo preserve 90% of energy: rank = {k_90}")
print(f"To preserve 95% of energy: rank = {k_95}")

## Part 6: Connection to Machine Learning

### Why This Matters for ML Engineers

SVD and low-rank approximation are fundamental to many ML techniques:

1. **Principal Component Analysis (PCA)**: PCA is essentially SVD! The principal components are the right singular vectors.

2. **Matrix Factorization for Recommender Systems**: Netflix Prize winning solutions used SVD-like techniques to factor the user-item matrix.

3. **Latent Semantic Analysis (LSA)**: Uses SVD to find "topics" in documents.

4. **Neural Network Compression**: SVD can compress weight matrices in neural networks.

5. **Data Preprocessing**: Dimensionality reduction before feeding data to ML models.

### Mathematical Insight: Why Does This Work?

The key theorem is the **Eckart-Young-Mirsky theorem**:

> The best rank-k approximation (in Frobenius or spectral norm) of a matrix A is given by truncating its SVD to the k largest singular values.

This is why SVD is optimal for compression!

## Part 7: Hands-On Exercise

Try these exercises to deepen your understanding:

1. **Load a color image** and compress each RGB channel separately using SVD. Compare results.

2. **Find the optimal rank** for your image by setting a target compression ratio (e.g., 10x) and finding the maximum rank that achieves it.

3. **Compare with JPEG**: Load a JPEG image and see how SVD compression compares at similar file sizes.

4. **Experiment with different images**: Natural images, faces, text, diagrams. Which compress better with SVD?

5. **Implement incremental reconstruction**: Show how the image improves as you add more singular values one by one.

In [ ]:
# Exercise: Incremental reconstruction visualization
def incremental_reconstruction_animation(U, S, Vt, ranks):
    """
    Show how image quality improves with more singular values.
    """
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    for idx, k in enumerate(ranks):
        img_k = low_rank_approximation(U, S, Vt, k)
        axes[idx].imshow(img_k, cmap='gray')
        axes[idx].set_title(f'Rank {k}', fontsize=11)
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.show()

# Try it!
incremental_ranks = [1, 2, 5, 10, 20, 30, 50, 100]
incremental_reconstruction_animation(U_img, S_img, Vt_img, incremental_ranks)

## Summary and Key Takeaways

### What We Learned

1. **SVD decomposes** any matrix into three matrices: $A = U\Sigma V^T$

2. **Low-rank approximation** keeps only the top $k$ singular values, compressing the data

3. **Compression-quality tradeoff**: More singular values = better quality but less compression

4. **Eckart-Young-Mirsky theorem**: SVD gives the optimal rank-k approximation

5. **ML connections**: SVD powers PCA, recommender systems, LSA, and more

### Next Steps

- Move on to **Project 2: PCA** to see how SVD is used for dimensionality reduction in ML
- Explore **Project 3: Recommender Systems** to see matrix factorization in action
- Read about the Netflix Prize and how SVD-based methods won the competition

### Further Reading

- [SVD Tutorial](https://web.mit.edu/be.400/www/SVD/Singular_Value_Decomposition.htm)
- [Image Compression with SVD](https://timbaumann.info/svd-image-compression-demo/)
- Boyd & Vandenberghe, "Convex Optimization" - Chapter 4

---

**Great work!** You now understand how SVD enables image compression and its connection to machine learning. 🎉